In [27]:
import os
import numpy as np
import pandas as pd
import pickle
import spacy

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.tag import pos_tag
from nltk.stem import WordNetLemmatizer
from nltk.probability import FreqDist
from nltk.classify import NaiveBayesClassifier, accuracy

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [28]:
DATASET_PATH = './jobpostingdata.csv'  # 1 = penipuan
MODEL_PATH = './naive_bayes_model.pkl'

stop_words = stopwords.words('english')
lemmatizer = WordNetLemmatizer()
vectorizer = TfidfVectorizer()

In [29]:
def load_dataset():
    df = pd.read_csv(DATASET_PATH)
    return df['title'], df['fraudulent'], df['text']

def get_tag(tag):
    if tag.startswith('J'):
        return 'a'
    elif tag.startswith('R'):
        return 'r'
    elif tag.startswith('V'):
        return 'v'
    else:
        return 'n'
    
def preprocess(text):
    text = text.lower()
    text = word_tokenize(text)
    text = [w for w in text if w not in stop_words]
    text = [w for w in text if w.isalpha()]
    
    tagged = pos_tag(text)
    text = [lemmatizer.lemmatize(w, get_tag(t)) for (w, t) in tagged]
    return ' '.join(text)

def extract_feature(freq_dist, text):
    words = text.split()
    return {word: (word in words) for word in freq_dist}

def train(text, labels):
    texts = text.apply(preprocess)
    all_words = ' '.join(texts).split()
    
    freq_dist = FreqDist(all_words)
    feature_set = [(extract_feature(freq_dist, text), label)
                   for (text, label) in zip(texts, labels)]
    split = int(0.8 * len(feature_set))
    train_set = feature_set[:split]
    test_set = feature_set[split:]
    
    classifier = NaiveBayesClassifier.train(train_set)
    print(f"Accuracy: {accuracy(classifier, test_set)}")
    
    vectorizer.fit_transform(texts)
    vectors = vectorizer.fit_transform([texts])
    
    return classifier, freq_dist, vectorizer, vectors

def train_or_load_model():
    if not os.path.exists(MODEL_PATH):
        titles, labels, texts = load_dataset()
        model = train(texts, labels)
        
        with open(MODEL_PATH, 'wb') as f:
            pickle.dump((*model, titles, texts), f)
        return (*model, titles, texts)
    else:
        with open(MODEL_PATH, 'rb') as f:
            pickle.load(f)

In [30]:
def menu():
    text = ''
    text_category = ''
    
    classifier, freq_dist, vectorizer, vectors, titles, texts = train_or_load_model()
    
    while True:
        print('Job Posting Classification')
        print('Your Text:', text if text else 'None')
        print('Your Text Category:')
        print('1. Write yout texts')
        print('2. View Recommendation')
        print('3. View NER')
        print('4. Exit')
        
        choice = input('>> ')
        if choice == '1':
            input_text = input('Write your text: ')
            if len(input_text) < 20 or len(input_text.split()) < 3:
                continue
            clean_text = preprocess(text)
            feature = extract_feature(freq_dist, clean_text)
            result = classifier.classify(feature)
            if result == 1:
                text_category = 'Fake Job Posting'
            else:
                text_category = 'Real Job Posting'
            
        elif choice == '2':
            if len(text) == 0:
                print('No Text')
                continue
            clean_text = preprocess(text)
            vector = vectorizer.transform(text)
            similar = cosine_similarity(vector, vectors)[0]
            top = np.argsort(similar)[-5:][::-1]
            
            for i in top:
                print('Title:', titles.iloc[i])
                print('Similarity:', similar[i])
                
        elif choice == '3':
            if len(text) == 0:
                print('No Text')
                continue
            nlp = spacy.load('en_core_web_sm')
            doc = nlp(text)
            
            allowed_ent = {"PERSON"}
            categories = {}
            for ent in doc.ents:
                if ent.label_ not in allowed_ent:
                    continue
                
                label = ent.label_
                if label not in categories:
                    categories[label] = []
                categories[label].append(ent.text)
            
        elif choice == '4':
            pass
        else:
            print('Invalid choice!')

In [31]:
menu()

AttributeError: partially initialized module 'nltk' has no attribute 'data' (most likely due to a circular import)